# [양자컴퓨팅 부트캠프] Day 2: 다중 큐비트, 텐서곱, 그리고 양자 얽힘(Entanglement)

이번 2부 실습에서는 1부에서 다룬 다중 큐비트 이론을 **순수 NumPy** 행렬 연산으로 직접 구현하고 검증합니다.

### 학습 목표 및 실습 구성 (75분)
1. **[Lab 1] Kronecker Product($\otimes$)**: 4차원 계산 기저 생성, 혼합곱 성질 검증, $H \otimes I$ 연산자 구성 및 유니터리성 확인
2. **[Lab 2] CNOT($\mathrm{CX}$) 게이트와 Bell 상태**: $4 \times 4$ $\mathrm{CX}$ 행렬 정의, Bell 상태 생성, 게이트 비가환성($AB \neq BA$) 검증
3. **[Lab 3] 2큐비트 순수 상태 비분리성 & 제어 반전 $\mathrm{CX}_{1\to 0}$**: 행렬식($\det(M) \neq 0$) 검정 및 제어-타깃 반전 행렬 도출
4. **[Lab 4] 부분 측정(Partial Measurement)과 재정규화**: 중첩 유지 케이스 및 Bell 상태 붕괴 시뮬레이션
5. **[Lab 5] 2큐비트 Shot 샘플링과 시각화**: 단일 실행 출력(단일 비트열) 확인 및 1,000 Shots 빈도 히스토그램 시각화
6. **[Lab 6] $n$-큐비트 확장(`kron_all`)과 3큐비트 GHZ vs W 상태**: $8 \times 8$ 연산자 조립 및 3큐비트 부분 측정 후 얽힘 유지 여부 검증

In [ ]:
from __future__ import annotations
import numpy as np
import matplotlib.pyplot as plt

# 기본 1큐비트 기저 및 유니터리 연산자 정의
KET_0 = np.array([[1.0], [0.0]], dtype=complex)
KET_1 = np.array([[0.0], [1.0]], dtype=complex)

GATE_I = np.eye(2, dtype=complex)
GATE_X = np.array([[0, 1], [1, 0]], dtype=complex)
GATE_Z = np.array([[1, 0], [0, -1]], dtype=complex)
GATE_H = (1.0 / np.sqrt(2.0)) * np.array([[1, 1], [1, -1]], dtype=complex)

print('1큐비트 기본 기저 및 연산자 세팅 완료!')

## [Lab 1] Kronecker Product($\otimes$), 혼합곱 성질, 그리고 $H \otimes I$

NumPy의 `np.kron(A, B)` 함수는 임의 크기 행렬의 Kronecker Product($A \otimes B$)를 계산합니다.
1. **과제 1-1**: 4차원 계산 기저 $|00\rangle, |01\rangle, |10\rangle, |11\rangle$를 생성하세요.
2. **과제 1-2**: 혼합곱 성질 $(H \otimes X)(Z \otimes H) = (HZ) \otimes (XH)$가 수치적으로 일치하는지 좌변(`lhs`)과 우변(`rhs`)을 각각 계산해 검증하세요.
3. **과제 1-3**: $q_0$에만 $H$를 적용하는 연산자 `H_I`($H \otimes I$)를 구성하고, 4대 계산 기저에 적용한 결과 및 유니터리성($U^* U = I_4$)을 확인하세요.

In [ ]:
# [실습 과제 1-1] np.kron을 사용하여 2큐비트 계산 기저 열벡터를 완성하세요.
KET_00 = np.kron(KET_0, KET_0)
KET_01 = None
KET_10 = None
KET_11 = None

# [실습 과제 1-2] 혼합곱 공식 (H ⊗ X)(Z ⊗ H) == (HZ) ⊗ (XH)의 좌변과 우변을 각각 작성하세요.
lhs = None
rhs = None

# [실습 과제 1-3] H_I (H ⊗ I) 행렬을 구성하세요.
H_I = None

# [자가 검증] 아래 코드를 실행해 확인하세요 (수정 불필요)
assert KET_01 is not None and np.allclose(KET_01, [[0], [1], [0], [0]]), 'KET_01 정의 오류!'
assert KET_10 is not None and np.allclose(KET_10, [[0], [0], [1], [0]]), 'KET_10 정의 오류!'
assert KET_11 is not None and np.allclose(KET_11, [[0], [0], [0], [1]]), 'KET_11 정의 오류!'
assert lhs is not None and rhs is not None and np.allclose(lhs, rhs), '혼합곱 공식 불일치!'
assert H_I is not None and H_I.shape == (4, 4), 'H_I는 4x4 행렬이어야 합니다!'
assert np.allclose(H_I.conj().T @ H_I, np.eye(4, dtype=complex)), 'H_I가 유니터리하지 않습니다!'
assert np.allclose(H_I @ KET_01, (1/np.sqrt(2)) * (KET_01 + KET_11)), '(H⊗I)|01> 결과 불일치!'
print('[Lab 1 성공] 4대 기저, 혼합곱 성질, H⊗I 유니터리성 및 기저 작용 검증 완료!')

## [Lab 2] $4 \times 4$ CNOT($\mathrm{CX}$) 게이트, Bell 상태, 그리고 비가환성

1부에서 유도한 2단계 Bell 회로의 전체 연산식은 다음과 같습니다:
$$|\Phi^+\rangle = \mathrm{CX} \cdot (H \otimes I)|00\rangle$$

1. **과제 2-1**: $q_0$이 제어, $q_1$이 타깃인 $4 \times 4$ 행렬 `GATE_CNOT`을 정의하세요.
2. **과제 2-2**: `psi_step1`($(H \otimes I)|00\rangle$)과 최종 Bell 상태 `psi_bell`($\mathrm{CX} \cdot (H \otimes I)|00\rangle$)을 계산하세요.
3. **과제 2-3 (비가환성 검증)**: 연산 순서를 뒤집어 `psi_reversed`($(H \otimes I) \cdot \mathrm{CX}|00\rangle$)를 계산하고, `psi_bell`과 서로 다름을 확인하세요.

In [ ]:
# [실습 과제 2-1] 4x4 CNOT 행렬을 정의하세요 (dtype=complex 권장).
GATE_CNOT = None

# [실습 과제 2-2] Bell 회로 2단계 연산을 순서대로 수행하세요.
psi_step1 = None
psi_bell = None

# [실습 과제 2-3] 게이트 순서를 반대로 적용한 (H ⊗ I) @ CNOT @ |00> 를 계산하세요.
psi_reversed = None

# [자가 검증] 1부 수식 계산 결과와 1:1 대조 (수정 불필요)
expected_bell = (1.0 / np.sqrt(2.0)) * (KET_00 + KET_11)
expected_rev = (1.0 / np.sqrt(2.0)) * (KET_00 + KET_10)
assert GATE_CNOT is not None and GATE_CNOT.shape == (4, 4), 'CNOT은 4x4 행렬이어야 합니다!'
assert np.allclose(GATE_CNOT.conj().T @ GATE_CNOT, np.eye(4)), 'CNOT이 유니터리하지 않습니다!'
assert psi_step1 is not None and np.allclose(psi_step1, expected_rev), '1단계 상태(|+0>) 오류!'
assert psi_bell is not None and np.allclose(psi_bell, expected_bell), '수식 계산 Bell 상태와 불일치!'
assert psi_reversed is not None and not np.allclose(psi_bell, psi_reversed), '비가환성 검증 실패!'
print('[Lab 2 성공] 수식 계산 |Phi+> 100% 일치 및 게이트 비가환성(CX @ H_I != H_I @ CX) 확인!')
print('최종 Bell 상태벡터:', psi_bell.flatten().real)

## [Lab 3] 2큐비트 순수 상태 비분리성 조건과 제어 반전 $\mathrm{CX}_{1\to 0}$

2큐비트 순수 상태 $|\psi\rangle = c_{00}|00\rangle + c_{01}|01\rangle + c_{10}|10\rangle + c_{11}|11\rangle$가 곱상태로 분리될 필요충분조건은 계수 행렬 $M = \begin{pmatrix} c_{00} & c_{01} \\ c_{10} & c_{11} \end{pmatrix}$의 행렬식이 0인 것입니다 ($\det(M) = c_{00}c_{11} - c_{01}c_{10} = 0$).

> **적용 범위 주의**: 위 $2 \times 2$ 행렬식 조건은 **2큐비트 순수 상태**에 한해 성립하는 특수 조건이며, 3큐비트 이상 다중 큐비트 얽힘에는 단일 행렬식 공식으로 일반화되지 않습니다.

1. **과제 3-1**: `is_entangled(state_4d)` 함수를 완성해 `psi_step1`, `psi_reversed`, `psi_bell`의 비분리성을 판별하세요.
2. **과제 3-2 (제어 반전 행렬 도출)**: 1부에서 참고로 다룬 **"$q_1$이 제어, $q_0$이 타깃인 반전 CNOT($\mathrm{CX}_{1\to 0}$)"**의 $4 \times 4$ 행렬 `GATE_CNOT_REV`를 직접 유도하여 작성하세요.

In [ ]:
# [실습 과제 3-1] 2큐비트 순수 상태 비분리성 확인 함수를 완성하세요.
def is_entangled(state_4d: np.ndarray, tol: float = 1e-9) -> tuple[bool, float]:
    """2큐비트 순수 상태벡터가 분리 불가능한 얽힘 상태인지 det = c00*c11 - c01*c10 조건으로 확인합니다."""
    v = state_4d.flatten()
    det = None
    entangled = not np.isclose(abs(det), 0.0, atol=tol)
    return entangled, float(abs(det))

# [실습 과제 3-2] q1이 제어(Control), q0이 타깃(Target)인 4x4 행렬을 직접 구성하세요.
# 힌트: |00>->|00>, |01>->|11>, |10>->|10>, |11>->|01>
GATE_CNOT_REV = None

# [자가 검증] (수정 불필요)
ent1, det1 = is_entangled(psi_step1)
ent_rev, det_rev = is_entangled(psi_reversed)
ent2, det2 = is_entangled(psi_bell)
assert not ent1 and np.isclose(det1, 0.0), 'Step 1 상태(|+0>)는 곱상태(det=0)여야 합니다.'
assert not ent_rev and np.isclose(det_rev, 0.0), '순서를 뒤집은 상태는 얽힘이 생성되지 않아야 합니다.'
assert ent2 and np.isclose(det2, 0.5), 'Bell 상태(|Phi+>)는 det=0.5인 최대 얽힘 상태여야 합니다.'

assert GATE_CNOT_REV is not None and GATE_CNOT_REV.shape == (4, 4), '4x4 행렬을 정의하세요!'
assert np.allclose(GATE_CNOT_REV @ KET_01, KET_11), 'q1=1일 때 q0이 반전(|01>->|11>)되어야 합니다!'
assert np.allclose(GATE_CNOT_REV @ KET_11, KET_01), 'q1=1일 때 q0이 반전(|11>->|01>)되어야 합니다!'
assert np.allclose(GATE_CNOT_REV @ KET_10, KET_10), 'q1=0일 때 상태가 유지(|10>->|10>)되어야 합니다!'
print(f'[Lab 3 성공] 곱상태 det={det1:.1f} / Bell det={det2:.4f} 및 반전 CX_(1->0) 행렬 검증 완료!')

## [Lab 4] 부분 측정(Partial Measurement)과 재정규화 시뮬레이션

1부에서 다룬 **다중 큐비트 Born 규칙(부분 측정)**을 함수로 구현합니다.
임의의 2큐비트 상태 $|\psi\rangle = c_{00}|00\rangle + c_{01}|01\rangle + c_{10}|10\rangle + c_{11}|11\rangle$에서 **1번 큐비트($q_0$)만 단독 측정**할 때:
* $q_0=0$ 관측 확률: $P(q_0=0) = |c_{00}|^2 + |c_{01}|^2$, 측정 후 $q_1$의 2차원 상태는 $\frac{1}{\sqrt{P(q_0=0)}} \begin{pmatrix} c_{00} \\ c_{01} \end{pmatrix}$로 재정규화
* $q_0=1$ 관측 확률: $P(q_0=1) = |c_{10}|^2 + |c_{11}|^2$, 측정 후 $q_1$의 2차원 상태는 $\frac{1}{\sqrt{P(q_0=1)}} \begin{pmatrix} c_{10} \\ c_{11} \end{pmatrix}$로 재정규화

구현한 함수를 1부에서 다룬 **중첩 유지 예시 상태**($\frac{1}{2}|00\rangle + \frac{1}{2}|01\rangle + \frac{1}{\sqrt{2}}|10\rangle$)와 **Bell 상태**($|\Phi^+\rangle$) 두 가지에 모두 적용해 확인하세요.

In [ ]:
# [실습 과제 4] 부분 측정 (Partial Measurement) 시뮬레이션
def simulate_partial_measurement(state_4d: np.ndarray) -> tuple[tuple[float, np.ndarray], tuple[float, np.ndarray]]:
    """q0 측정 시 결과별 확률과 측정 직후 q1의 재정규화된 2차원 상태벡터를 반환합니다."""
    v = state_4d.flatten()
    p0 = float(abs(v[0])**2 + abs(v[1])**2)
    p1 = float(abs(v[2])**2 + abs(v[3])**2)
    
    # p0 > 0 이면 [v[0], v[1]] / sqrt(p0), p1 > 0 이면 [v[2], v[3]] / sqrt(p1) 로 재정규화하세요.
    state_q1_0 = None
    state_q1_1 = None
    return (p0, state_q1_0), (p1, state_q1_1)

# [자가 검증 1] 1부 예시 상태: |psi> = 0.5|00> + 0.5|01> + (1/sqrt(2))|10>
psi_example = 0.5 * KET_00 + 0.5 * KET_01 + (1.0 / np.sqrt(2.0)) * KET_10
(ex_p0, ex_s0), (ex_p1, ex_s1) = simulate_partial_measurement(psi_example)
assert np.isclose(ex_p0, 0.5) and np.isclose(ex_p1, 0.5)
assert ex_s0 is not None and np.allclose(ex_s0, [1/np.sqrt(2), 1/np.sqrt(2)]), 'q0=0 관측 시 q1은 |+> 중첩을 유지해야 합니다!'
assert ex_s1 is not None and np.allclose(ex_s1, [1.0, 0.0]), 'q0=1 관측 시 q1은 |0>이어야 합니다!'

# [자가 검증 2] Bell 상태: |Phi+>
(p0, s0), (p1, s1) = simulate_partial_measurement(psi_bell)
assert s0 is not None and np.allclose(s0, [1, 0]), 'q0=0 관측 시 q1은 |0>으로 붕괴해야 합니다!'
assert s1 is not None and np.allclose(s1, [0, 1]), 'q0=1 관측 시 q1은 |1>으로 붕괴해야 합니다!'

print('[Lab 4 성공] 예시 상태(중첩 유지) 및 Bell 상태(즉각 붕괴) 부분 측정 검증 완료!')
print(f'- 예시 상태 q0=0 관측({ex_p0*100:.0f}%): q1 = {ex_s0.real.round(3)} (|+> 유지)')
print(f'- Bell 상태 q0=0 관측({p0*100:.0f}%): q1 = {s0.real}, q0=1 관측({p1*100:.0f}%): q1 = {s1.real}')

## [Lab 5] 2큐비트 Shot 샘플링과 제어된 상관관계 시각화

양자 회로를 **1회 실행(1 Shot)**하면 확률 숫자가 아니라 단 하나의 고전 비트열(`'00'`, `'01'`, `'10'`, `'11'` 중 하나)이 출력됩니다.
`np.random.choice`를 사용하여 `sample_2qubit_measurements` 함수를 완성하고,
* `shots=1`일 때의 단일 비트열 출력
* `shots=1000`일 때 **완전 랜덤 동전 2개(`|++>`)** vs **제어된 주사위(`|Phi+>`)**의 빈도 분포 차이
를 막대 그래프로 비교해 봅시다.

In [ ]:
# [실습 과제 5] 2큐비트 Born 규칙 기반 Shot 샘플러 구현
def sample_2qubit_measurements(state_4d: np.ndarray, shots: int = 1000) -> dict[str, int]:
    """4차원 상태벡터에서 Born 확률 |c_x|^2에 따라 shots번 비트열을 샘플링하여 빈도 딕셔너리를 반환합니다."""
    labels = ['00', '01', '10', '11']
    probs = np.abs(state_4d.flatten()) ** 2
    probs = probs / np.sum(probs)  # 부동소수점 오차 정규화
    
    # np.random.choice(labels, size=shots, p=probs)를 이용해 샘플링 후 각 라벨의 출현 횟수를 counts에 집계하세요.
    samples = None
    counts = {lbl: 0 for lbl in labels}
    # samples의 각 원소 빈도를 counts에 누적하세요.
    return counts

# [자가 검증 & 시각화]
single_shot = sample_2qubit_measurements(psi_bell, shots=1)
assert sum(single_shot.values()) == 1, 'shots=1일 때 총 관측 횟수는 1이어야 합니다!'
observed_bitstring = [k for k, v in single_shot.items() if v == 1][0]
assert observed_bitstring in ('00', '11'), 'Bell 상태에서는 00 또는 11만 관측되어야 합니다!'
print(f'[1 Shot 실행 결과] 실제 계측기 출력 비트열: {observed_bitstring}')

# 비교 대상: 분리 상태 |++> = (H ⊗ H)|00> vs 얽힘 상태 |Phi+>
psi_plus_plus = np.kron(GATE_H, GATE_H) @ KET_00
counts_random = sample_2qubit_measurements(psi_plus_plus, shots=1000)
counts_bell = sample_2qubit_measurements(psi_bell, shots=1000)
assert counts_bell['01'] == 0 and counts_bell['10'] == 0, 'Bell 상태에서 01, 10이 나오면 안 됩니다!'

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
axes[0].bar(counts_random.keys(), counts_random.values(), color='#6c757d')
axes[0].set_title('Separable |++> (Independent Dice, 1000 Shots)')
axes[0].set_ylabel('Counts')
axes[1].bar(counts_bell.keys(), counts_bell.values(), color='#0072ce')
axes[1].set_title('Entangled |Phi+> (Controlled Dice, 1000 Shots)')
plt.tight_layout()
plt.show()
print('[Lab 5 성공] 독립 상태(|++>)와 제어된 상관관계(|Phi+>)의 1,000 Shots 비교 시각화 완료!')

## [Lab 6] $n$-큐비트 텐서곱 일반화(`kron_all`)와 3큐비트 GHZ vs W 상태 검증

이제 2큐비트($\mathbb{C}^4$)를 넘어 **3큐비트($\mathbb{C}^8$, $8 \times 8$ 행렬)** 및 임의의 $n$큐비트 연산으로 확장합니다.

1. **과제 6-1**: `functools.reduce`와 `np.kron`을 이용해 임의 개수의 벡터/행렬을 순서대로 텐서곱하는 `kron_all(*mats)` 함수를 구현하세요.
2. **과제 6-2**: `kron_all`로 $8 \times 8$ 연산자 $H \otimes I \otimes I$, $\mathrm{CX} \otimes I$, $I \otimes \mathrm{CX}$를 구성하여 $|000\rangle$로부터 **GHZ 상태** `psi_ghz`$=\frac{|000\rangle+|111\rangle}{\sqrt{2}}$를 합성하세요.
3. **과제 6-3**: GHZ 상태와 W 상태($|\mathrm{W}\rangle = \frac{|001\rangle+|010\rangle+|100\rangle}{\sqrt{3}}$)에서 각각 $q_0=0$을 부분 측정한 뒤 남은 2큐비트(4차원) 상태에 **Lab 3의 `is_entangled` 함수**를 적용해 얽힘 소멸(GHZ) vs 얽힘 유지(W)를 검증하세요.

In [ ]:
from functools import reduce

# [실습 과제 6-1] 임의 개수의 행렬/벡터를 순서대로 텐서곱하는 kron_all 함수를 완성하세요.
def kron_all(*mats: np.ndarray) -> np.ndarray:
    return None

# [실습 과제 6-2] 3큐비트 연산자(8x8) 및 GHZ 상태 합성
ket_000 = kron_all(KET_0, KET_0, KET_0)
H_II = None   # H (x) I (x) I
CX_I = None   # CX (x) I
I_CX = None   # I (x) CX
psi_ghz = None

# W 상태 정의: (|001> + |010> + |100>) / sqrt(3)
psi_w = (
    kron_all(KET_0, KET_0, KET_1)
    + kron_all(KET_0, KET_1, KET_0)
    + kron_all(KET_1, KET_0, KET_0)
) / np.sqrt(3.0)

# [실습 과제 6-3] 3큐비트 상태에서 q0=0 관측 후 남은 (q1, q2) 4차원 상태벡터 반환
def collapse_q0_zero_3q(state_8d: np.ndarray) -> np.ndarray:
    v = state_8d.flatten()
    # q0=0에 해당하는 앞 4개 진폭 v[:4]를 추출하고 norm 1로 재정규화하세요.
    return None

# [자가 검증] (수정 불필요)
assert ket_000 is not None and ket_000.shape == (8, 1), 'ket_000은 8x1 벡터여야 합니다!'
assert H_II is not None and H_II.shape == (8, 8)
assert psi_ghz is not None and np.allclose(psi_ghz, (kron_all(KET_0, KET_0, KET_0) + kron_all(KET_1, KET_1, KET_1)) / np.sqrt(2.0)), 'GHZ 상태 오류!'

ghz_after_0 = collapse_q0_zero_3q(psi_ghz)
w_after_0 = collapse_q0_zero_3q(psi_w)
assert ghz_after_0 is not None and w_after_0 is not None

ent_ghz, det_ghz = is_entangled(ghz_after_0)
ent_w, det_w = is_entangled(w_after_0)
assert not ent_ghz and np.isclose(det_ghz, 0.0), 'GHZ는 q0 측정 후 나머지 2큐비트가 곱상태여야 합니다!'
assert ent_w and np.isclose(det_w, 0.5), 'W 상태는 q0=0 측정 후 나머지 2큐비트가 Bell 상태로 얽혀 있어야 합니다!'

print(f'[Lab 6 성공] GHZ 붕괴 후 det={det_ghz:.1f} (얽힘 소멸) / W 붕괴 후 det={det_w:.4f} (Bell 얽힘 유지)!')

---
### 2일차 총정리 & 3일차 예고 (다중 큐비트와 양자 얽힘)

오늘 2일차 이론과 실습에서는 1일차의 단일 큐비트 모델($\mathbb{C}^2$)을 다중 큐비트 복소 벡터 공간($\mathbb{C}^{2^n}$)으로 확장하고, **순수 NumPy**만으로 얽힘의 수학적 구조를 구현했습니다.

#### 오늘 정리한 핵심 원리
1. **상태공간의 지수적 확장 ($2^n$)과 다중 큐비트 Born 규칙**
   - $n$큐비트의 복합 상태는 $2^n$차원 복소 벡터 공간의 단위벡터($\sum |c_x|^2 = 1$)로 기술됩니다.
   - 전체 측정 시 출력은 확률값이 아닌 단일 고전 비트열이며, 부분 측정 시 남은 성분은 관측 확률의 제곱근으로 나누어 다시 크기 1로 재정규화됩니다.

2. **Kronecker Product($\otimes$)와 2큐비트 상호작용 게이트 (CNOT / $\mathrm{CX}$)**
   - 독립된 연산은 $A \otimes B$로 결합되며, 혼합곱 성질 $(A\otimes B)(C\otimes D) = (AC)\otimes(BD)$이 성립합니다.
   - 큐비트 간 상호작용을 만드는 대표 게이트 $\mathrm{CX}$와 $H \otimes I$를 결합해 Bell 상태 $|\Phi^+\rangle = \frac{|00\rangle+|11\rangle}{\sqrt{2}}$를 생성했으며, 연산 순서에 따른 비가환성($\mathrm{CX}(H\otimes I) \neq (H\otimes I)\mathrm{CX}$)을 확인했습니다.

3. **양자 얽힘, No-Signaling, 그리고 제어된 상관관계**
   - Bell 상태는 단일 큐비트 상태들의 텐서곱으로 분해할 수 없는 비분리 상태입니다.
   - 고전 통신 채널 없이는 초광속 정보 전달이 불가능하지만(No-Signaling), 얽힘은 두 큐비트 사이에 정교한 상관관계를 만들어 양자 우위와 확률 분포 제어를 가능하게 합니다.

---
#### 3일차 예고: Qiskit 회로 실습과 Grover 알고리즘
> "신은 주사위 놀음을 하지만, 제어된 주사위 놀이를 한다."
>
> 다음 시간에는 글로벌 표준 양자 프레임워크인 **Qiskit**을 도입하여 양자 회로를 직접 조립하고, 위상 반전(Oracle)과 간섭 증폭(Diffusion)을 이용해 비정렬 탐색 속도를 획기적으로 높이는 **Grover 알고리즘**을 탐구합니다.